In [4]:
# ── Cell 1: Title ─────────────────────────────────────────────
# MedAssist AI — Notebook 1: Data Exploration & Indexing
# ─────────────────────────────────────────────────────────────
# What this notebook does:
#   1. Loads the PubMed QA dataset (500 abstracts)
#   2. Explores the data structure and statistics
#   3. Chunks documents for retrieval
#   4. Builds and saves the FAISS vector index
#
# No GPU needed — runs entirely on CPU
# Runtime: ~2 minutes

In [1]:
# ── Cell 2: Install ───────────────────────────────────────────
!pip install -q --force-reinstall "numpy==1.26.4"
!pip install -q --force-reinstall "pandas==2.2.2"
!pip install -q "langchain==0.2.16" "langchain-community==0.2.16"
!pip install -q "langchain-huggingface==0.0.3" "faiss-cpu==1.8.0"
!pip install -q "sentence-transformers==3.0.1" "datasets==2.21.0"
!pip install -q "tqdm==4.66.5"
# ── Step 4: CRITICAL — restart the kernel to flush the broken numpy ─────────
print("⚠️  Restarting kernel to apply new numpy version...")
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
kaggle-environments 1.27.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.

{'status': 'ok', 'restart': True}

In [1]:
# ── Cell 3: Imports ───────────────────────────────────────────
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

CONFIG = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "max_abstracts"  : 500,
    "chunk_size"     : 512,
    "chunk_overlap"  : 64,
    "top_k_retrieval": 3,
}
print("✅ Imports ready")

✅ Imports ready


In [2]:
# ── Cell 4: Load Dataset ──────────────────────────────────────
dataset = load_dataset("pubmed_qa", "pqa_labeled", split="train")
print(f"✅ Loaded {len(dataset)} records")

# ── Explore structure ─────────────────────────────────────────
sample = dataset[0]
print(f"\nQUESTION  : {sample['question']}")
print(f"DECISION  : {sample['final_decision']}")
print(f"ANSWER    : {sample['long_answer'][:200]}...")

# ── Distribution of decisions ─────────────────────────────────
df = pd.DataFrame(dataset)
print(f"\n📊 Decision distribution:")
print(df["final_decision"].value_counts())

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

✅ Loaded 1000 records

QUESTION  : Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
DECISION  : yes
ANSWER    : Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of ...

📊 Decision distribution:
final_decision
yes      552
no       338
maybe    110
Name: count, dtype: int64


In [3]:
# ── Cell 5: Preprocess and Chunk ──────────────────────────────
def records_to_documents(dataset, max_samples=500):
    """Convert PubMed QA records to LangChain Document objects."""
    documents = []
    for i, record in enumerate(tqdm(dataset, desc="Loading")):
        if i >= max_samples:
            break
        ctx  = record.get("context", {})
        text = " ".join(ctx.get("contexts", [])) \
               if isinstance(ctx, dict) else str(ctx)
        if len(text.strip()) < 50:
            continue
        documents.append(Document(
            page_content=text,
            metadata={
                "doc_id"     : i,
                "question"   : record.get("question", ""),
                "gold_answer": record.get("long_answer", "")[:200],
                "decision"   : record.get("final_decision", ""),
                "source"     : f"PubMedQA_record_{i}",
            }
        ))
    return documents

raw_docs = records_to_documents(dataset, max_samples=500)
print(f"✅ {len(raw_docs)} documents loaded")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CONFIG["chunk_size"],
    chunk_overlap=CONFIG["chunk_overlap"],
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(raw_docs)
print(f"✅ {len(chunks)} chunks created")
print(f"📏 Avg chunk length: {np.mean([len(d.page_content) for d in chunks]):.0f} chars")

Loading:  50%|█████     | 500/1000 [00:00<00:00, 5881.74it/s]

✅ 500 documents loaded
✅ 1743 chunks created
📏 Avg chunk length: 386 chars


In [4]:
# ── Cell 6: Build and Save FAISS Index ───────────────────────
embedding_model = HuggingFaceEmbeddings(
    model_name=CONFIG["embedding_model"],
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 64},
)

print(f"📐 Embedding dim: {len(embedding_model.embed_query('test'))}")

vectorstore = FAISS.from_documents(chunks, embedding_model)
print(f"✅ FAISS index built — {vectorstore.index.ntotal} vectors")

vectorstore.save_local("/kaggle/working/faiss_medassist_index")
print("💾 Index saved → /kaggle/working/faiss_medassist_index")

2026-03-16 00:39:41.121689: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773621581.454573     223 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773621581.542190     223 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773621582.280241     223 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773621582.280289     223 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773621582.280292     223 computation_placer.cc:177] computation placer alr

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

📐 Embedding dim: 384
✅ FAISS index built — 1743 vectors
💾 Index saved → /kaggle/working/faiss_medassist_index


In [6]:
# ── Cell 7: Test Retrieval ────────────────────────────────────
queries = [
    "What are the effects of metformin on blood glucose?",
    "Does programmed cell death occur in plant leaves?",
]

for query in queries:
    results = vectorstore.similarity_search_with_score(query, k=3)
    print(f"\n❓ {query}")
    for i, (doc, l2_score) in enumerate(results):

        # Convert L2 distance → 0-1 similarity for readability
        # similarity = 1 / (1 + l2_score)
        # score=0.0  → similarity=1.00 (perfect)
        # score=0.61 → similarity=0.62 (good)
        # score=1.0  → similarity=0.50 (moderate)
        # score=2.0  → similarity=0.33 (weak)
        similarity = 1 / (1 + l2_score)

        print(f"  [{i+1}] L2={l2_score:.2f}  similarity={similarity:.2f}")
        print(f"        {doc.page_content[:100].strip()}...")
        print(f"        source: {doc.metadata.get('source','?')}")


❓ What are the effects of metformin on blood glucose?
  [1] L2=0.84  similarity=0.54
        To evaluate the effects of insulin 30/70 twice daily or bedtime isophane (NPH) insulin plus continue...
        source: PubMedQA_record_381
  [2] L2=0.89  similarity=0.53
        . A total of 64 insulin-naive patients treated with maximal feasible dosages of sulfonylurea and met...
        source: PubMedQA_record_381
  [3] L2=0.99  similarity=0.50
        Hypoglycaemia caused by glucose-lowering therapy has been linked to cardiovascular (CV) events. The...
        source: PubMedQA_record_97

❓ Does programmed cell death occur in plant leaves?
  [1] L2=0.61  similarity=0.62
        Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Apon...
        source: PubMedQA_record_0
  [2] L2=1.00  similarity=0.50
        . However, the cold-induced modifications in the cell wall properties were more pronounced in the le...
        source: PubMedQA_record_265
 